In [1]:
# Name: Cheryl Cook
# ECS Username: cookcher | Student ID: 300587497

# Part 3: Neural Networks

import numpy as np
import torch

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset

# Part 3: Neural Networks

# Data Loading and Preprocessing

# Load the Digits dataset
digits = load_digits()
X = digits.data
y = digits.target

# Set a random seed for reproducibility
torch.manual_seed(231)
np.random.seed(231)

# Split the data into training and test sets (80%-20%, random state=231)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=231)

# Normalise the images by scaling pixel values to 0-1 range
normaliser = MinMaxScaler()
# normaliser is fitted to the training data
X_train_normalised = normaliser.fit_transform(X_train)
# normaliser fit to training set applied to test set
X_test_normalised = normaliser.transform(X_test)

# Convert the numpy arrays into PyTorch tensors using torch.from_numpy from Tutorial 8
# Tensors allow for multi-dimensional data rep. & efficient computation
# .float() for features - as NN computations are more precise with floats
X_train_tensor = torch.from_numpy(X_train_normalised).float()
# .long() for target values, as Pytorch loss expects 'long' format
y_train_tensor = torch.from_numpy(y_train).long()
X_test_tensor = torch.from_numpy(X_test_normalised).float()
y_test_tensor = torch.from_numpy(y_test).long()

# Create DataLoader objects for both training and test sets
"""
Batch size 64 is a good balance between small batch sizes that introduce
too much noise, and computationally inefficient large batches with
reduced noise that can lead to overfitting.
64 batch size provides enough variability to aid generalisation,
and enough samples to calculate stable gradient estimates for smoother
training.
"""
batch_size = 64
# TensorDataset() bundles multiple tensors into a single dataset
training_dataset = TensorDataset(X_train_tensor, y_train_tensor)
"""
DataLoader: divides the dataset into smaller batches of batch_size,
allows for more frequent weight updates (once per batch).
For training data, shuffle=true to improve generalisation,
as randomness prevents the model from learning artifical
order-related patterns from the data.
"""
training_loader = DataLoader(training_dataset, batch_size=batch_size, shuffle=True)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
"""
For test data, shuffle=false as shuffling test data would introduce
randomness that isn't reflective of how the model would be used in real-world scenarios.
Keeping the test data fixed ensures both consistent evaluation across several runs,
and fair performance comparisons between different hyperparameters or models.
"""
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [2]:
# Set a random seed for reproducibility
torch.manual_seed(231)
np.random.seed(231)
"""
(a) Define a neural network class by extending torch.nn.Module.
NN should have one input layer, one hidden layer of 128 neurons, and one output layer.
Use the ReLU activation function for the hidden layer,
and the softmax activation function for the output layer.
You determine the number of neurons in the input and output layers.
"""
import torch.nn.functional as F

# NeuralNetwork inherits from torch.nn.Module
class NeuralNetwork(torch.nn.Module):
  def __init__(self):
    # super() calls the torch.nn.Module constructor
    super().__init__()
    # Define the three layers using torch.nn.Linear (creates fully connected/linear layers)
    # Input layer has 64 neurons (64 features) and transforms that input data into 128 neurons
    self.input_layer = torch.nn.Linear(64, 128)
    # Hidden layer with 128 neurons in and out
    self.hidden_layer = torch.nn.Linear(128, 128)
    # Output layer with 10 neurons out that correspond to the 10 classes (digits 0-9)
    self.output_layer = torch.nn.Linear(128, 10)

  # Forward pass through the neural network
  def forward(self, x):
    """
    Pass the input data (x) through the input layer,
    where it undergeoes a linear transformation to z,
    which is the sum of a weighted input feature
    and a bias term. (z = w * x + b)
    Then apply ReLU activation (max(z, 0)),
    which replaces any negative values with 0 and
    preserves positive values.
    """
    x = F.relu(self.input_layer(x))
    # Apply linear transformation and ReLU activation to hidden layer
    x = F.relu(self.hidden_layer(x))
    """
    Apply linear transformation and softmax function to output layer.
    Softmax converts raw output scores into probabilities for each of
    the possible output classes.
    """
    softmax = torch.nn.Softmax(dim=1)
    x = softmax(self.output_layer(x))
    return x

In [3]:
# Set a random seed for reproducibility
torch.manual_seed(231)
np.random.seed(231)
"""
(b) Use the torch.nn.CrossEntropyLoss for your loss function,
choose an optimizer from torch.optim.SGD and torch.optim.Adam
and set an appropriate learning rate.
Train the model for 15 epochs.
After each epoch, print the training loss and accuracy.
"""

#Given a defined NN structure, how to train (optimize) it:
def train_show(network, lossFunc, optimiser, epochs):

    for epoch in range(epochs):
      # Storing every calculated loss and accuracy in the epoch to calculate average
      lossHistory = []
      accuHistory = []
      """
      training mode has active dropout - meaning neurons are randomly
      dropped/zeroed out during each forward pass to make the model more robust
      and prevent overfitting
      """
      network.train() # Set model to training mode
      for data, targ in training_loader:
        """
        Gradients in PyTorch accumulate by default,
        so need to zero out the gradients of all parameters before
        starting forward pass to avoid mixing gradients between batches.
        """
        optimiser.zero_grad()

        # perform forward pass by calling forward() in NeuralNetwork class
        y = network.forward(data)
        # calculate the loss
        loss = lossFunc(y,targ)
        """
        Computes gradients of the loss with respect to all model parameters
        using backpropagation. This calculates how each parameter contributed
        to the loss and is used for optimisation.
        """
        loss.backward()            # runs autograd, to get the gradients needed by optimiser
        """
        Updates the model parameters in the direction that minimises the loss
        using the gradients calculated in the backward pass.
        """
        optimiser.step()           # take a step

        """
        torch.argmax(y,dim=1) returns the index of the class with the highest
        probability for each input sample in the batch, which is then compared
        to the actual (targ) class value for that sample. A boolean tensor is
        returned to indicate whether the prediction is correct or not,
        and .float() converts that boolean tensor into a float tensor
        (1 for correct, 0 for incorrect).
        torch.mean() calculates the mean of all the input samples' float tensors
        to effectively get the proportion of correct predictions for that batch.
        """
        accuracy = torch.mean((torch.argmax(y,dim=1) == targ).float())
        # add loss and accuracy to lists for calculating average over epoch later
        # extracts the loss value as a Python scalar
        lossHistory.append(loss.detach().item())
        # detaches the accuracy tensor from the computation graph
        accuHistory.append(accuracy.detach())

      # Calculate average loss and accuracy for the epoch
      avg_loss = sum(lossHistory) / len(lossHistory)
      avg_accuracy = sum(accuHistory) / len(accuHistory)

      # Print average training loss and accuracy over the epoch
      print(f"Epoch {epoch+1}: Loss = {avg_loss:.3f}, Accuracy = {int(avg_accuracy*100)}%")

# Instantiate the neural network using the previously defined NeuralNetwork class
model = NeuralNetwork()

# Define the loss function
lossFunction = torch.nn.CrossEntropyLoss() # calculates how off predictions are from actual values

# Choose an optimiser and an appropriate learning rate
optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
# Adam has adaptive learning rate for each parameter
# lr: parameters adjusted by amount proportional to lr * the loss gradient with respect to those parameters

print("Training Output:")
train_show(model, lossFunction, optimiser, 15)

Training Output:
Epoch 1: Loss = 2.288, Accuracy = 41%
Epoch 2: Loss = 2.166, Accuracy = 59%
Epoch 3: Loss = 1.888, Accuracy = 66%
Epoch 4: Loss = 1.733, Accuracy = 80%
Epoch 5: Loss = 1.664, Accuracy = 84%
Epoch 6: Loss = 1.631, Accuracy = 86%
Epoch 7: Loss = 1.580, Accuracy = 93%
Epoch 8: Loss = 1.559, Accuracy = 94%
Epoch 9: Loss = 1.544, Accuracy = 94%
Epoch 10: Loss = 1.531, Accuracy = 95%
Epoch 11: Loss = 1.522, Accuracy = 96%
Epoch 12: Loss = 1.518, Accuracy = 96%
Epoch 13: Loss = 1.510, Accuracy = 97%
Epoch 14: Loss = 1.512, Accuracy = 96%
Epoch 15: Loss = 1.502, Accuracy = 97%


In [4]:
# Set a random seed for reproducibility
torch.manual_seed(231)
np.random.seed(231)
"""
(c) Evaluate the model on the test set to measure its accuracy. Print the test
accuracy and show five example predictions along with their actual labels.
"""
def test_network(network):
  """
  eval() mode disables dropout, so it uses all neurons; also
  uses running estimates of mean and variance collected during model
  training to normalise the data (rather than mini-batch stats);
  this ensures that the normalisation is based on the entire training set
  """
  network.eval() # Set the network to evaluation mode
  # Storing every calculate loss and accuracy to calculate the average
  lossHistory = []
  accuHistory = []
  # Disable gradient computation
  with torch.no_grad():
    for test_data, test_targ in test_loader:
      # Peform forward pass
      y_test = network(test_data)
      # Compute the loss
      test_loss = lossFunction(y_test, test_targ)
      # Compute the accuracy
      test_accuracy = torch.mean((torch.argmax(y_test,dim=1) == test_targ).float())

      # add loss and accuracy to lists to calculate average later
      lossHistory.append(test_loss.detach().item())
      accuHistory.append(test_accuracy.detach())

    # Calculate average loss and accuracy for the test set
    avg_loss = sum(lossHistory) / len(lossHistory)
    avg_accuracy = sum(accuHistory) / len(accuHistory)

    # Print the test accuracy
    print(f"Test Accuracy: {int(torch.round(avg_accuracy * 100).item())}%")

    # Choose five examples with their predictions and actual labels

    # Extracting the predictions of the first five test samples
    example_predictions = torch.argmax(y_test, dim=1)[:5]
    example_labels = test_targ[:5]

    # Print the predictions vs actual labels for those five samples
    for i in range(5):
        print(f"Test Image {i+1}: Predicted Label = {example_predictions[i].item()}, Actual Label = {example_labels[i].item()}")

test_network(model)

Test Accuracy: 95%
Test Image 1: Predicted Label = 5, Actual Label = 5
Test Image 2: Predicted Label = 3, Actual Label = 3
Test Image 3: Predicted Label = 1, Actual Label = 1
Test Image 4: Predicted Label = 2, Actual Label = 2
Test Image 5: Predicted Label = 9, Actual Label = 9


In [5]:
"""
(d) Evaluate and compare the effectiveness of different activation functions
including Sigmoid and Tanh in place of ReLU.
"""

# Modifying NeuralNetwork() to have __init__ accept an activation function as a parameter
# Referring to this class as NewNeuralNetwork()
# Set a random seed for reproducibility
torch.manual_seed(231)
np.random.seed(231)

import torch.nn.functional as F

class NewNeuralNetwork(torch.nn.Module):
  def __init__(self, activation_function):
    super().__init__()
    # Define the three layers using torch.nn.Linear (creates fully connected/linear layers)
    # Input layer has 64 neurons (64 features) and transforms that input data into 128 neurons
    self.input_layer = torch.nn.Linear(64, 128)
    # Hidden layer with 128 neurons in and out
    self.hidden_layer = torch.nn.Linear(128, 128)
    # Output layer with 10 neurons out that correspond to the 10 classes (digits 0-9)
    self.output_layer = torch.nn.Linear(128, 10)
    self.activation_function = activation_function

  def forward(self, x):
    # Forward pass through the neural network
    # Apply linear transformation and activation function to input layer
    x = self.activation_function(self.input_layer(x))
    # Apply linear transformation and activation function to hidden layer
    x = self.activation_function(self.hidden_layer(x))
    # Apply linear transformation and softmax activation to the output layer
    softmax = torch.nn.Softmax(dim=1)
    x = softmax(self.output_layer(x))
    return x



In [6]:
"""
(d) Evaluate and compare the effectiveness of different activation functions
including Sigmoid and Tanh in place of ReLU.
"""

# Set a random seed for reproducibility
torch.manual_seed(231)
np.random.seed(231)

# Store the activation functions in a list to loop through and compare results
"""
Activation functions introduce non-linearity into the model.
ReLU: (max(z, 0)), replaces any negative values with 0
and preserves positive values; activates only the neurons
with positive outputs to make the model sparse and efficient

Sigmoid: Maps any input z to a value between 0 and 1, which can be
interpreted as probabilities

Tanh: Maps any input z to a value between -1 and 1, makes it zero-centered
which can help center the data and make optimisation easier

ReLU and Tanh often used in hidden layers, Sigmoid in output layers for binary
classfication
"""
activation_functions = {
    'ReLU': F.relu,
    'Sigmoid': torch.sigmoid,
    'Tanh': torch.tanh
}

# Train and test the network with each activation function
for name, act_fnc in activation_functions.items():
    print(f"\nTraining and testing with {name} activation function")
    # initialising neural network using NewNeuralNetwork class
    new_model = NewNeuralNetwork(act_fnc)
    # retaining same optimiser and learning rate as before
    new_optimiser = torch.optim.Adam(new_model.parameters(), lr=0.001)
    # retaining same loss function as before
    new_lossFunc = torch.nn.CrossEntropyLoss()
    # train_show will print all 15 epoch results
    train_show(new_model, new_lossFunc, new_optimiser, 15)
    # test_network will print average accuracy and five sample results
    test_network(new_model)


Training and testing with ReLU activation function
Epoch 1: Loss = 2.288, Accuracy = 41%
Epoch 2: Loss = 2.166, Accuracy = 59%
Epoch 3: Loss = 1.888, Accuracy = 66%
Epoch 4: Loss = 1.733, Accuracy = 80%
Epoch 5: Loss = 1.664, Accuracy = 84%
Epoch 6: Loss = 1.631, Accuracy = 86%
Epoch 7: Loss = 1.580, Accuracy = 93%
Epoch 8: Loss = 1.559, Accuracy = 94%
Epoch 9: Loss = 1.544, Accuracy = 94%
Epoch 10: Loss = 1.531, Accuracy = 95%
Epoch 11: Loss = 1.522, Accuracy = 96%
Epoch 12: Loss = 1.518, Accuracy = 96%
Epoch 13: Loss = 1.510, Accuracy = 97%
Epoch 14: Loss = 1.512, Accuracy = 96%
Epoch 15: Loss = 1.502, Accuracy = 97%
Test Accuracy: 95%
Test Image 1: Predicted Label = 5, Actual Label = 5
Test Image 2: Predicted Label = 3, Actual Label = 3
Test Image 3: Predicted Label = 1, Actual Label = 1
Test Image 4: Predicted Label = 2, Actual Label = 2
Test Image 5: Predicted Label = 9, Actual Label = 9

Training and testing with Sigmoid activation function
Epoch 1: Loss = 2.304, Accuracy = 9%
E